# SDOH / neighborhood / environmental exposure -- concept discovery

Searches the real CDR for candidate concept_ids covering the phenotype list below,
rather than guessing -- this repo's own history is full of plausible-looking
concept_ids that turned out wrong when actually checked (`hip_circumference`'s LOINC
mapped to *thigh* circumference; the PPI alcohol-frequency items were unpopulated
under their own vocabulary's concept_ids and had to be traced to what they actually
"Maps to"). Same concept-then-count pattern `01_query_filter_check.ipynb`'s
appendices already use: search `{CDR}.concept` by name pattern, then count real rows
per candidate against `{CDR}.observation`, so a plausible name with zero real data
doesn't get mistaken for a usable phenotype.

| Domain | Phenotype |
|---|---|
| Individual SES | years of education, household income, employment, insurance |
| Financial | food insecurity, financial strain |
| Social | loneliness, social support, perceived stress |
| Neighborhood | deprivation index, median income, poverty, education level, uninsured fraction |
| Environment | PM2.5, NO2, ozone, NDVI, noise exposure |

**Individual SES/Financial/Social** are almost certainly AoU's "Basics" survey
(income/employment/education/insurance) and "Social Determinants of Health" (SDOH)
survey module (food insecurity, financial strain, loneliness -- likely the UCLA-3
scale, perceived stress -- likely PSS) -- both real PPI-vocabulary survey
instruments, but their exact concept_ids need confirming here, not assuming.

**Neighborhood** overlaps with `zip3_ses_map`, which `residualize_phenotypes.ipynb`'s
`pull_covariates()` already reads for `median_income`/`poverty`/`deprivation_index`
-- checks that table's full column list below for `education level`/
`uninsured fraction` before assuming they need a separate source.

**Environment** (PM2.5/NO2/ozone/NDVI/noise) -- not assumed to exist in this CDR
version at all; searches `INFORMATION_SCHEMA.TABLES` directly rather than guessing
a table name.

All output is aggregate concept metadata + counts, never person-level, same
convention as every other appendix in this repo.

## Compute resource

Small aggregate BigQuery queries only -- Workbench 2.0's default (2 CPU / 13 GB) is enough.

In [ ]:
required_pkgs <- c("dplyr", "readr", "stringr", "bigrquery", "allofus")
missing_pkgs <- required_pkgs[!sapply(required_pkgs, requireNamespace, quietly = TRUE)]
if (length(missing_pkgs) > 0) install.packages(missing_pkgs)

library(dplyr)
library(readr)
library(stringr)
library(bigrquery)
library(allofus)

con <- aou_connect()
run_query <- function(sql) collect(aou_sql(sql))

## Individual SES / Financial / Social: search PPI survey concepts

Searches `{CDR}.concept` (vocabulary_id = `'PPI'`, the AoU survey vocabulary) for
each phenotype's keyword pattern(s), then counts real `{CDR}.observation` rows per
candidate concept_id -- same pattern as `01_query_filter_check.ipynb`'s lifestyle
appendix. `concept_code` is included since AoU's PPI codes (e.g. `Employment_*`,
`Insurance_*`, `SDOH_*`) are often a clearer signal than the free-text
`concept_name` alone for telling which survey module an item belongs to.

Keyword patterns below are a starting point, not confirmed -- inspect the results
and narrow/broaden them if a phenotype's real item doesn't show up, same as this
repo's existing lifestyle/waist-hip appendices needed real iteration to land on
the right concept_ids.

In [ ]:
PPI_SEARCH_TERMS <- list(
  years_of_education    = c("%education%", "%grade%school%"),
  household_income       = c("%income%"),
  employment              = c("%employ%"),
  insurance                = c("%insurance%"),
  food_insecurity          = c("%food%worr%", "%food%afford%", "%food%insecur%", "%hungry%"),
  financial_strain         = c("%financial%", "%money%", "%unable to pay%"),
  loneliness                = c("%lonel%", "%isolat%"),
  social_support            = c("%social support%", "%emotional support%", "%someone%support%"),
  perceived_stress          = c("%perceived stress%", "%stress%")
)

search_ppi_concept <- function(phenotype_name, patterns) {
  like_clauses <- paste(sprintf("LOWER(concept_name) LIKE '%s'", tolower(patterns)), collapse = " OR ")
  concepts <- run_query(sprintf("
    SELECT concept_id, concept_name, concept_code, domain_id, vocabulary_id, standard_concept
    FROM {CDR}.concept
    WHERE vocabulary_id = 'PPI'
      AND (%s)
  ", like_clauses))
  if (nrow(concepts) == 0) {
    return(tibble(phenotype_name, concept_id = NA_integer_, concept_name = NA_character_,
                   concept_code = NA_character_, n_persons = NA_integer_, n_rows = NA_integer_))
  }

  counts <- bind_rows(lapply(concepts$concept_id, function(cid) {
    run_query(sprintf("
      SELECT COUNT(DISTINCT person_id) AS n_persons, COUNT(*) AS n_rows
      FROM {CDR}.observation
      WHERE observation_concept_id = %s
    ", cid)) %>% mutate(concept_id = cid, .before = 1)
  }))

  concepts %>%
    inner_join(counts, by = "concept_id") %>%
    mutate(phenotype_name, .before = 1) %>%
    arrange(desc(n_persons))
}

ppi_results <- bind_rows(lapply(names(PPI_SEARCH_TERMS), function(name) {
  search_ppi_concept(name, PPI_SEARCH_TERMS[[name]])
}))

ppi_results %>% arrange(phenotype_name, desc(n_persons))

## Neighborhood: check `zip3_ses_map`'s full schema

`residualize_phenotypes.ipynb`'s `pull_covariates()` already reads `median_income`/
`fraction_poverty`/`deprivation_index` from this table -- checks whether
`education level`/`uninsured fraction` are already columns here (no new join
needed) before assuming they need a separate source.

In [ ]:
zip3_columns <- run_query("
  SELECT column_name, data_type
  FROM {CDR}.INFORMATION_SCHEMA.COLUMNS
  WHERE table_name = 'zip3_ses_map'
  ORDER BY ordinal_position
")
zip3_columns

## Environment: search for any linked exposure table

Not assumed to exist -- searches `{CDR}.INFORMATION_SCHEMA.TABLES` for any table
whose name suggests environmental/exposure data (PM2.5, NO2, ozone, NDVI, noise,
"environment", "exposure", "air quality"), rather than guessing a table name that
might not exist in this CDR version at all. If nothing turns up, that's a real,
useful negative result -- these variables would need an external
geocoded/zip-linked dataset joined in outside the CDR, not something missed by a
narrower search.

In [ ]:
ENV_TABLE_PATTERNS <- c("%environ%", "%exposure%", "%air_qual%", "%pm25%", "%pm2_5%",
                        "%ozone%", "%no2%", "%ndvi%", "%noise%", "%greenspace%")

like_clauses <- paste(sprintf("LOWER(table_name) LIKE '%s'", ENV_TABLE_PATTERNS), collapse = " OR ")
env_tables <- run_query(sprintf("
  SELECT table_name
  FROM {CDR}.INFORMATION_SCHEMA.TABLES
  WHERE (%s)
", like_clauses))

if (nrow(env_tables) == 0) {
  message("No table names matching environmental-exposure patterns found in this CDR's dataset. ",
          "This likely means PM2.5/NO2/ozone/NDVI/noise data isn't linked in this CDR version -- ",
          "would need an external geocoded dataset (e.g. EPA AQS, CDC/ATSDR environmental justice ",
          "index) joined on zip3/geography outside the CDR, not something this search missed.")
} else {
  message(sprintf("%d candidate table(s) found -- inspect their schemas next", nrow(env_tables)))
}
env_tables

In [ ]:
# Run only if sdoh-env-search found candidates above -- introspects each
# candidate table's columns so you can tell what geography/pollutant/units it
# actually carries before trying to join it to anything
if (nrow(env_tables) > 0) {
  env_table_columns <- bind_rows(lapply(env_tables$table_name, function(tbl) {
    run_query(sprintf("
      SELECT column_name, data_type
      FROM {CDR}.INFORMATION_SCHEMA.COLUMNS
      WHERE table_name = '%s'
      ORDER BY ordinal_position
    ", tbl)) %>% mutate(table_name = tbl, .before = 1)
  }))
  env_table_columns
}

## Confirmed findings so far (real query results, not assumptions)

- **Employment**: `1585952` (Employment: Employment Status) has zero rows under
  its own `observation_concept_id`; the real data lives under standard concept
  `40771090` with `observation_source_concept_id = 1585952` -- but the answer
  distribution is a flat set of unordered categories (employed/retired/student/
  unable-to-work/...), not ordinal. **Dropped as a phenotype** -- no clean
  numeric or ordered-categorical signal.
- **Education level** (`1585940`): populated (742,427 rows) but purely
  categorical (`value_as_concept_id`, `value_as_number` always null) -- would
  need a manual ordinal recode (grade school -> doctorate) to use as a
  quantitative covariate. See `docs/phenotype_list.tsv` decision below.
- **Neighborhood SES**: `ds_zip_code_socioeconomic` is the *same* data as
  `zip3_ses_map` (identical columns), just exported as a dataset-builder
  person-level view (`PERSON_ID` + `OBSERVATION_DATETIME` directly) instead of
  a standalone zip3-keyed lookup table. Not a new source.
- **Environment** (PM2.5/NO2/ozone/NDVI/noise): confirmed real null -- no
  matching table beyond the two SES exports above. No linked environmental
  exposure data in this CDR version.
- **SES/education as covariates, not phenotypes**: both are classic
  confounders for heritability estimation (correlated with genotype via
  population structure/assortative mating, and with most outcome phenotypes) --
  belongs in `02_residualize_phenotypes.ipynb`'s covariate set alongside
  age/sex/PCs, not in the phenotype list.

## SDOH survey module: full item inventory

AoU has a dedicated SDOH survey module (`sdoh_*`-prefixed `concept_code`s,
distinct from core PPI/Basics) carrying five validated psychometric scales:
Cohen's Perceived Stress Scale (`cpss`, 10 items), RAND MOS Social Support
(`mos_ss`, 8 items), UCLA Loneliness Scale (`ucla_ls8`, 8 items), Everyday
Discrimination Scale (`eds`, 9 items), and Discrimination in Medical Setting
(`dms`, 7 items). None of these have a pre-computed composite score under their
own "instrument" concept_id (`1333217`/`1333283` return zero rows) -- only the
individual items are populated, so composite scores need to be summed here.

In [ ]:
sdoh_module_concepts <- run_query("
  SELECT concept_id, concept_name, concept_code
  FROM {CDR}.concept
  WHERE REGEXP_CONTAINS(concept_code, r'^sdoh_')
    AND vocabulary_id = 'PPI'
  ORDER BY concept_code
")
sdoh_module_concepts

## PSS / Social Support / Loneliness: composite scores

All three scales are `value_as_concept_id`-coded, not numeric -- confirmed by
checking one item from each (`cpss_3`, `mos_ss_1`, `ucla_ls8_2`): every
`value_as_number` was null, and `903096` ("PMI: Skip") showed up as a
non-response code in all three, to be treated as missing rather than mapped to
a Likert value. The three scales use **different label vocabularies**, so each
needs its own map (checked directly against `{CDR}.concept`, not assumed from
the instrument's published response options):

- **PSS** (`cpss_*`): Never(0) / Almost never(1) / Sometimes(2) / Fairly
  often(3) / Very often(4). Cohen's PSS-10 reverse-scores 4 positively-worded
  items (`cpss_4`, `cpss_5`, `cpss_7`, `cpss_8`) as `4 - x` before summing --
  applied below, since skipping this would understate validity against the
  published instrument.
- **MOS Social Support** (`mos_ss_*`): None of the time(0) / A little of the
  time(1) / Some of the time(2) / Most of the time(3) / All of the time(4).
- **UCLA Loneliness** (`ucla_ls8_*`): Never(0) / Rarely(1) / Sometimes(2) /
  Often(3) -- a 4-point scale in this CDR, not the original instrument's
  3-point scale; flagged since it deviates from published UCLA-LS norms.

Composite = sum of mapped item scores per person, requiring all items for that
scale to be non-missing (no partial-scale imputation). `eds`/`dms`
(discrimination scales) are deliberately not computed here -- open question
whether they belong as phenotypes or as environmental-exposure covariates,
same reasoning as the SES/education covariate decision above.

In [ ]:
# Likert value_as_concept_id -> integer score, confirmed against {CDR}.concept
# labels above. "903096" (PMI: Skip) maps to NA in every scale -- non-response,
# not a real answer.
LIKERT_MAPS <- list(
  cpss     = c(`45876662` = 0, `45881665` = 1, `45882528` = 2, `45879226` = 3, `45884601` = 4, `903096` = NA_real_),
  mos_ss   = c(`45884592` = 0, `45876996` = 1, `45879198` = 2, `45879199` = 3, `45883772` = 4, `903096` = NA_real_),
  ucla_ls8 = c(`45876662` = 0, `45876672` = 1, `45882528` = 2, `45884455` = 3, `903096` = NA_real_)
)
# Cohen's PSS-10 reverse-scored (positively-worded) items: 4 - x, not x
PSS_REVERSE_ITEMS <- c("cpss_4", "cpss_5", "cpss_7", "cpss_8")
SCALE_N_ITEMS <- c(cpss = 10, mos_ss = 8, ucla_ls8 = 8)

item_concepts <- sdoh_module_concepts %>%
  filter(str_detect(concept_code, "^(cpss|mos_ss|ucla_ls8)_[0-9]+$")) %>%
  mutate(scale = str_extract(concept_code, "^(cpss|mos_ss|ucla_ls8)"), item = concept_code)

sdoh_item_obs <- run_query(sprintf("
  SELECT person_id, observation_source_concept_id, value_as_concept_id
  FROM {CDR}.observation
  WHERE observation_source_concept_id IN (%s)
", paste(item_concepts$concept_id, collapse = ",")))

score_item <- function(scale, value_as_concept_id) {
  mapply(function(s, v) {
    m <- LIKERT_MAPS[[s]]
    key <- as.character(v)
    if (is.null(m) || !(key %in% names(m))) return(NA_real_)
    unname(m[key])
  }, scale, value_as_concept_id)
}

sdoh_item_scored <- sdoh_item_obs %>%
  inner_join(item_concepts %>% select(concept_id, scale, item), by = c("observation_source_concept_id" = "concept_id")) %>%
  mutate(
    raw_score = score_item(scale, value_as_concept_id),
    score = if_else(item %in% PSS_REVERSE_ITEMS, 4 - raw_score, raw_score)
  )

sdoh_composites <- sdoh_item_scored %>%
  filter(!is.na(score)) %>%
  group_by(person_id, scale) %>%
  summarise(n_items = n_distinct(item), total_score = sum(score), .groups = "drop") %>%
  mutate(n_expected = SCALE_N_ITEMS[scale]) %>%
  filter(n_items == n_expected) %>%  # require the full scale, no partial-scale imputation
  select(person_id, scale, total_score)

pss_scores             <- sdoh_composites %>% filter(scale == "cpss")     %>% select(person_id, pss_score = total_score)
social_support_scores  <- sdoh_composites %>% filter(scale == "mos_ss")   %>% select(person_id, social_support_score = total_score)
loneliness_scores      <- sdoh_composites %>% filter(scale == "ucla_ls8") %>% select(person_id, loneliness_score = total_score)

sdoh_wide <- pss_scores %>%
  full_join(social_support_scores, by = "person_id") %>%
  full_join(loneliness_scores, by = "person_id")

sdoh_wide %>%
  summarise(
    n_pss = sum(!is.na(pss_score)),
    n_social_support = sum(!is.na(social_support_score)),
    n_loneliness = sum(!is.na(loneliness_score))
  )

## Summary

**Phenotypes (candidates for `docs/phenotype_list.tsv`, `source = "survey_composite"`):**
`pss_score` (0-40, Cohen's PSS-10, 4 items reverse-scored), `social_support_score`
(0-32, RAND MOS-SS8), `loneliness_score` (0-24, UCLA-LS8 4-point variant) --
computed in `sdoh_wide` above, `n_expected` items required non-missing per person
(no partial-scale imputation).

**Covariates, not phenotypes (add to `02_residualize_phenotypes.ipynb`'s
covariate set alongside age/sex/PCs):** neighborhood SES
(`zip3_ses_map`/`ds_zip_code_socioeconomic` -- already partially wired in),
education level (`1585940`, needs a manual ordinal recode -- categorical only,
no numeric field). Both are classic confounders for heritability estimation
(correlated with genotype via population structure/assortative mating, and with
most outcome phenotypes), not phenotypes to estimate heritability *of*.

**Dropped:** employment status (`1585952`/`40771090`) -- populated but flat
unordered categories, no clean numeric/ordinal signal.

**Open decision:** Everyday Discrimination (`eds_*`) and Discrimination in
Medical Setting (`dms_*`) -- structurally identical to PSS/social-support (same
composite-scoring approach would apply) but conceptually closer to an
environmental-exposure covariate than a heritable trait; not computed here
pending that call.

**Confirmed real null:** environmental exposure data (PM2.5/NO2/ozone/NDVI/noise)
-- no linked table in this CDR version; would need an external geocoded dataset
joined on zip3 outside the CDR.